In [ ]:
import json
from google.colab import files

uploaded = files.upload()

In [ ]:
file_name = "Input_JSON_Schedule_Optimization.json"

with open(file_name, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Machines:", len(data["machines"]))
print("Products:", len(data["products"]))
print("Production requests:", len(data["product_requests"]))

In [ ]:
# Each time slot represents 5 minutes.
slot_minutes = 5

# Use the same operation order as the Excel pilot.
selected_modes = [
    "Sublimation TM3",
    "Anti-Shrinkage TM2",
    "Ironing TM1",
]

# Match each operation mode to its data.
modes_by_id = {
    mode["id"]: mode
    for mode in data["task_modes"]
}

# Build one continuous energy profile for the three operations.
combined_energy_wh = []

for mode_name in selected_modes:
    energy_profile = modes_by_id[mode_name]["power"]

    duration_minutes = len(energy_profile) * slot_minutes
    energy_kwh = sum(energy_profile) / 1000

    combined_energy_wh.extend(energy_profile)

    print(
        f"{mode_name}: "
        f"{duration_minutes} min | "
        f"{energy_kwh:.3f} kWh"
    )

total_duration_minutes = len(combined_energy_wh) * slot_minutes
total_energy_kwh = sum(combined_energy_wh) / 1000

print()
print(f"Total duration: {total_duration_minutes} min")
print(f"Total energy: {total_energy_kwh:.3f} kWh")

In [ ]:
# Read the grid electricity prices from the dataset.
grid_source = next(
    source
    for source in data["energy_sources"]
    if source["id"] == "Socket Energy"
)

# Dataset prices are expressed in EUR per Wh.
prices_eur_per_wh = grid_source["price"]

# The first working day starts at 07:00.
workday_start_minutes = 7 * 60
production_start_minutes = 9 * 60

# Find the price index corresponding to 09:00.
start_slot = (
    production_start_minutes - workday_start_minutes
) // slot_minutes

# Match each consumption interval to its electricity price.
costs_eur = []

for offset, energy_wh in enumerate(combined_energy_wh):
    price = prices_eur_per_wh[start_slot + offset]
    costs_eur.append(energy_wh * price)

baseline_cost_eur = sum(costs_eur)

print(f"Start slot: {start_slot}")
print(f"Total energy: {total_energy_kwh:.3f} kWh")
print(f"Electricity cost: {baseline_cost_eur:.8f} EUR")

# Check the result against the Excel pilot.
excel_baseline_cost_eur = 0.47016221

assert abs(
    baseline_cost_eur - excel_baseline_cost_eur
) < 0.00000001, "Python and Excel results do not match."

print("Check passed: Python matches Excel.")

In [ ]:
# First working day: 07:00–23:00.
workday_end_minutes = 23 * 60

working_slots = (
    workday_end_minutes - workday_start_minutes
) // slot_minutes

first_day_prices = prices_eur_per_wh[:working_slots]
job_slots = len(combined_energy_wh)

# Keep every operation inside the working day.
last_start_slot = working_slots - job_slots

results = []

for start_slot in range(last_start_slot + 1):
    cost_eur = sum(
        energy_wh * first_day_prices[start_slot + offset]
        for offset, energy_wh in enumerate(combined_energy_wh)
    )

    start_minutes = (
        workday_start_minutes + start_slot * slot_minutes
    )

    results.append({
        "start_minutes": start_minutes,
        "finish_minutes": start_minutes + total_duration_minutes,
        "cost_eur": cost_eur,
    })

# Select the lowest-cost candidate.
best_plan = min(results, key=lambda plan: plan["cost_eur"])


def format_time(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


savings_percent = (
    (baseline_cost_eur - best_plan["cost_eur"])
    / baseline_cost_eur
    * 100
)

print(f"Schedules evaluated: {len(results)}")
print(f"Best start: {format_time(best_plan['start_minutes'])}")
print(f"Finish: {format_time(best_plan['finish_minutes'])}")
print(f"Lowest cost: {best_plan['cost_eur']:.8f} EUR")
print(f"Savings versus 09:00: {savings_percent:.2f}%")

assert len(results) == 173
assert best_plan["start_minutes"] == 15 * 60 + 20
assert abs(best_plan["cost_eur"] - 0.43114033) < 0.00000001

print("Check passed: best plan matches Excel.")

In [ ]:
def find_best_plan_before(deadline_minutes):
    feasible_plans = [
        plan
        for plan in results
        if plan["finish_minutes"] <= deadline_minutes
    ]

    if not feasible_plans:
        return None

    return min(feasible_plans, key=lambda plan: plan["cost_eur"])


deadline_hours = [12, 15, 17, 23, 8]
deadline_results = []

for hour in deadline_hours:
    deadline_minutes = hour * 60
    plan = find_best_plan_before(deadline_minutes)

    deadline_results.append({
        "deadline_minutes": deadline_minutes,
        "best_plan": plan,
    })

    print(f"Deadline: {format_time(deadline_minutes)}")

    if plan is None:
        print("No feasible schedule\n")
        continue

    saving_percent = (
        (baseline_cost_eur - plan["cost_eur"])
        / baseline_cost_eur
        * 100
    )

    print(f"  Start: {format_time(plan['start_minutes'])}")
    print(f"  Finish: {format_time(plan['finish_minutes'])}")
    print(f"  Cost: {plan['cost_eur']:.6f} EUR")
    print(f"  Savings versus 09:00: {saving_percent:.2f}%\n")


# Check the 15:00 deadline against Excel.
plan_15 = find_best_plan_before(15 * 60)

assert plan_15 is not None
assert plan_15["start_minutes"] == 13 * 60 + 20
assert plan_15["finish_minutes"] <= 15 * 60
assert abs(plan_15["cost_eur"] - 0.45714978) < 0.00000001

# Check that the impossible deadline is recognised.
assert find_best_plan_before(8 * 60) is None

print("Checks passed: deadline scenarios match Excel.")

In [ ]:
import pandas as pd

# Convert all 173 candidate schedules into a table.
results_df = pd.DataFrame(results)

results_df["start_time"] = results_df["start_minutes"].map(format_time)
results_df["finish_time"] = results_df["finish_minutes"].map(format_time)

display(
    results_df[["start_time", "finish_time", "cost_eur"]]
    .head(10)
)

# Save the complete table without rounding the underlying results.
results_df.to_csv("pilot_start_time_results.csv", index=False)

print(f"Saved {len(results_df)} candidate schedules.")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    results_df["start_minutes"],
    results_df["cost_eur"],
    label="Fixed operation order",
)

ax.scatter(
    best_plan["start_minutes"],
    best_plan["cost_eur"],
    color="red",
    zorder=3,
    label=f"Lowest cost: {format_time(best_plan['start_minutes'])}",
)

ax.scatter(
    9 * 60,
    baseline_cost_eur,
    color="black",
    zorder=3,
    label="09:00 baseline",
)

ticks = list(range(7 * 60, 23 * 60, 60))
ax.set_xticks(ticks)
ax.set_xticklabels([format_time(t) for t in ticks], rotation=45)

ax.set_xlabel("Production start time")
ax.set_ylabel("Electricity cost (EUR)")
ax.set_title("Pilot scheduling costs — first working day")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig("pilot_start_time_cost.png", dpi=300)
plt.show()

In [ ]:
import zipfile

with zipfile.ZipFile("pilot_results.zip", "w") as archive:
    archive.write("pilot_start_time_results.csv")
    archive.write("pilot_start_time_cost.png")

files.download("pilot_results.zip")

In [ ]:
import json
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))
data = json.loads(uploaded[file_name].decode("utf-8"))

print("Dataset loaded.")
print("Machines:", len(data["machines"]))
print("Products:", len(data["products"]))

In [ ]:
import pandas as pd

# Map each task to its available processing modes.
tasks_by_id = {
    task["id"]: task
    for task in data["tasks"]
}

# Find which machines can execute each mode.
machines_by_mode = {}

for machine in data["machines"]:
    for mode_name in machine["task_modes"]:
        machines_by_mode.setdefault(mode_name, []).append(
            machine["id"]
        )

# Extract the required tasks for every product.
product_task_rows = []

for product in data["products"]:
    for position, task_entry in enumerate(
        product["tasks"], start=1
    ):
        task_name = task_entry["task"]
        available_modes = tasks_by_id[task_name]["task_modes"]

        compatible_machines = sorted({
            machine_id
            for mode_name in available_modes
            for machine_id in machines_by_mode.get(mode_name, [])
        })

        product_task_rows.append({
            "product": product["id"],
            "listed_position": position,
            "required_task": task_name,
            "runs": task_entry["runs"],
            "available_modes": ", ".join(available_modes),
            "compatible_machines": ", ".join(compatible_machines),
        })

product_tasks_df = pd.DataFrame(product_task_rows)

# Display a small selection first.
example_products = [
    "PRINTED RIBBON",
    "PRINTED RIBBON  PES",
    "BRAIDED ELASTIC",
    "SMOOTH STRAP(A)",
]

display(
    product_tasks_df[
        product_tasks_df["product"].isin(example_products)
    ].reset_index(drop=True)
)

In [ ]:
referenced_modes = {
    mode_name
    for task in data["tasks"]
    for mode_name in task["task_modes"]
}

unassigned_modes = sorted(
    referenced_modes - set(machines_by_mode)
)

print("Modes without an assigned machine:", unassigned_modes)

In [ ]:
# Define three separate production jobs.
pilot_selection = [
    ("J1", "PRINTED RIBBON  PES", "Sublimation TM3"),
    ("J2", "BRAIDED ELASTIC", "Anti-Shrinkage TM2"),
    ("J3", "SMOOTH STRAP(A)", "Ironing TM1"),
]

products_by_id = {
    product["id"]: product
    for product in data["products"]
}

modes_by_id = {
    mode["id"]: mode
    for mode in data["task_modes"]
}

pilot_machine = next(
    machine
    for machine in data["machines"]
    if machine["id"] == "MAQ120"
)

pilot_jobs = []

for job_id, product_name, mode_name in pilot_selection:
    product = products_by_id[product_name]

    # Verify that this product requires exactly one task, once.
    assert len(product["tasks"]) == 1
    task_entry = product["tasks"][0]
    assert task_entry["runs"] == 1

    # Verify that the selected mode can perform the required task.
    task = tasks_by_id[task_entry["task"]]
    assert mode_name in task["task_modes"]

    # Verify that the selected machine supports this mode.
    assert mode_name in pilot_machine["task_modes"]

    energy_profile = modes_by_id[mode_name]["power"]

    pilot_jobs.append({
        "job_id": job_id,
        "product": product_name,
        "production_units": 1,
        "task": task_entry["task"],
        "mode": mode_name,
        "machine": "MAQ120",
        "duration_min": len(energy_profile) * 5,
        "energy_kwh": sum(energy_profile) / 1000,
    })

pilot_jobs_df = pd.DataFrame(pilot_jobs)

display(pilot_jobs_df)

print("Total duration:", pilot_jobs_df["duration_min"].sum(), "min")
print("Total energy:", pilot_jobs_df["energy_kwh"].sum(), "kWh")
print("Checks passed: products, tasks, modes and machine agree.")

In [ ]:
from itertools import permutations
import pandas as pd

slot_minutes = 5
day_start = 7 * 60
day_end = 23 * 60
deadline_minutes = 17 * 60

# Read the first working day's electricity prices.
grid_source = next(
    source
    for source in data["energy_sources"]
    if source["id"] == "Socket Energy"
)

day_prices = grid_source["price"][:192]


def time_label(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


# Keep all candidates so we can test other deadlines later.
job_schedule_results = []

for job_order in permutations(pilot_jobs):
    profile_wh = []

    for job in job_order:
        profile_wh.extend(
            modes_by_id[job["mode"]]["power"]
        )

    duration_minutes = len(profile_wh) * slot_minutes
    last_start_slot = len(day_prices) - len(profile_wh)

    for start_slot in range(last_start_slot + 1):
        start_minutes = day_start + start_slot * slot_minutes
        finish_minutes = start_minutes + duration_minutes

        cost_eur = sum(
            energy_wh * day_prices[start_slot + offset]
            for offset, energy_wh in enumerate(profile_wh)
        )

        job_schedule_results.append({
            "job_order": " → ".join(
                job["job_id"] for job in job_order
            ),
            "mode_order": " → ".join(
                job["mode"] for job in job_order
            ),
            "start_minutes": start_minutes,
            "finish_minutes": finish_minutes,
            "cost_eur": cost_eur,
        })

# Only retain schedules that meet the deadline.
feasible_schedules = [
    plan
    for plan in job_schedule_results
    if plan["finish_minutes"] <= deadline_minutes
]

print("All candidates:", len(job_schedule_results))
print("Candidates meeting deadline:", len(feasible_schedules))

if not feasible_schedules:
    print("No feasible schedule.")
else:
    best_job_plan = min(
        feasible_schedules,
        key=lambda plan: plan["cost_eur"],
    )

    print()
    print("Job order:", best_job_plan["job_order"])
    print("Operations:", best_job_plan["mode_order"])
    print("Start:", time_label(best_job_plan["start_minutes"]))
    print("Finish:", time_label(best_job_plan["finish_minutes"]))
    print(f"Cost: {best_job_plan['cost_eur']:.8f} EUR")

    assert best_job_plan["finish_minutes"] <= deadline_minutes
    assert abs(
        best_job_plan["cost_eur"] - 0.42738425
    ) < 0.00000001

    print("Check passed for the 17:00 deadline.")

In [ ]:
demand_rows = []

for job in pilot_jobs:
    requested_units = sum(
        request["amount"]
        for request in data["product_requests"]
        if request["product"] == job["product"]
    )

    demand_rows.append({
        "product": job["product"],
        "requested_units": requested_units,
        "total_duration_min": (
            requested_units * job["duration_min"]
        ),
        "total_energy_kwh": (
            requested_units * job["energy_kwh"]
        ),
    })

demand_df = pd.DataFrame(demand_rows)
display(demand_df)

required_minutes = demand_df["total_duration_min"].sum()
required_energy = demand_df["total_energy_kwh"].sum()
capacity_minutes = 16 * 60

print(f"Required machine time: {required_minutes} min")
print(f"Available machine time: {capacity_minutes} min")
print(f"Remaining time: {capacity_minutes - required_minutes} min")
print(f"Total energy: {required_energy:.3f} kWh")

In [ ]:
# Expand the selected product demands into individual jobs.
production_jobs = []

for template in pilot_jobs:
    quantity = sum(
        request["amount"]
        for request in data["product_requests"]
        if request["product"] == template["product"]
    )

    assert quantity == int(quantity) and quantity >= 0

    for unit_number in range(1, int(quantity) + 1):
        production_jobs.append({
            "job_id": f"{template['job_id']}_{unit_number:02d}",
            "product": template["product"],
            "mode": template["mode"],
            "energy_profile_wh": list(
                modes_by_id[template["mode"]]["power"]
            ),
        })

# Read prices for the first working day.
grid_source = next(
    source
    for source in data["energy_sources"]
    if source["id"] == "Socket Energy"
)

demand_day_prices = grid_source["price"][:192]

# Schedule the jobs consecutively, starting at 07:00.
current_slot = 0
demand_baseline_rows = []

for job in production_jobs:
    profile = job["energy_profile_wh"]
    finish_slot = current_slot + len(profile)

    assert finish_slot <= len(demand_day_prices), (
        "Production does not fit inside the working day."
    )

    cost_eur = sum(
        energy_wh * demand_day_prices[current_slot + offset]
        for offset, energy_wh in enumerate(profile)
    )

    start_minutes = 7 * 60 + current_slot * 5
    finish_minutes = 7 * 60 + finish_slot * 5

    demand_baseline_rows.append({
        "job_id": job["job_id"],
        "product": job["product"],
        "start": f"{start_minutes // 60:02d}:{start_minutes % 60:02d}",
        "finish": f"{finish_minutes // 60:02d}:{finish_minutes % 60:02d}",
        "energy_kwh": sum(profile) / 1000,
        "cost_eur": cost_eur,
    })

    current_slot = finish_slot

demand_baseline_df = pd.DataFrame(demand_baseline_rows)
demand_baseline_cost_eur = demand_baseline_df["cost_eur"].sum()

display(demand_baseline_df)

print("Jobs:", len(production_jobs))
print("Production time:", current_slot * 5, "min")
print("Finish:", demand_baseline_df.iloc[-1]["finish"])
print(
    f"Total energy: "
    f"{demand_baseline_df['energy_kwh'].sum():.3f} kWh"
)
print(f"Baseline cost: {demand_baseline_cost_eur:.6f} EUR")

assert len(production_jobs) == 29
assert current_slot * 5 == 905
assert abs(demand_baseline_cost_eur - 3.86335060) < 0.00000001

print("Checks passed.")

In [ ]:
from functools import lru_cache
from math import inf, isfinite

# Processing profiles for the three selected product types.
dp_profiles = [
    list(modes_by_id[job["mode"]]["power"])
    for job in pilot_jobs
]

dp_durations = [
    len(profile)
    for profile in dp_profiles
]

# Count the required production units of each type.
dp_quantities = tuple(
    sum(
        job["mode"] == template["mode"]
        for job in production_jobs
    )
    for template in pilot_jobs
)

dp_prices = list(demand_day_prices)
dp_horizon = len(dp_prices)

# Calculate the cost of starting each job type at each time.
dp_start_costs = []

for profile in dp_profiles:
    costs = []

    for start in range(dp_horizon + 1):
        if start + len(profile) > dp_horizon:
            costs.append(inf)
        else:
            costs.append(sum(
                energy_wh * dp_prices[start + offset]
                for offset, energy_wh in enumerate(profile)
            ))

    dp_start_costs.append(costs)


@lru_cache(maxsize=None)
def minimum_remaining_cost(time_slot, remaining):
    # Nothing remains to be produced.
    if not any(remaining):
        return 0.0

    # Reject states with insufficient time.
    required_slots = sum(
        quantity * duration
        for quantity, duration in zip(
            remaining, dp_durations
        )
    )

    if time_slot + required_slots > dp_horizon:
        return inf

    # Option 1: wait for five minutes.
    best_cost = minimum_remaining_cost(
        time_slot + 1, remaining
    )

    # Other options: produce one unit of an unfinished type.
    for job_type, quantity in enumerate(remaining):
        if quantity == 0:
            continue

        next_remaining = list(remaining)
        next_remaining[job_type] -= 1

        candidate_cost = (
            dp_start_costs[job_type][time_slot]
            + minimum_remaining_cost(
                time_slot + dp_durations[job_type],
                tuple(next_remaining),
            )
        )

        best_cost = min(best_cost, candidate_cost)

    return best_cost


optimized_demand_cost_eur = minimum_remaining_cost(
    0, dp_quantities
)

if not isfinite(optimized_demand_cost_eur):
    print("No feasible schedule.")
else:
    demand_savings_eur = (
        demand_baseline_cost_eur
        - optimized_demand_cost_eur
    )

    demand_savings_percent = (
        demand_savings_eur
        / demand_baseline_cost_eur
        * 100
    )

    print("Required quantities:", dp_quantities)
    print(f"Baseline: {demand_baseline_cost_eur:.8f} EUR")
    print(f"Optimized: {optimized_demand_cost_eur:.8f} EUR")
    print(f"Savings: {demand_savings_eur:.8f} EUR")
    print(f"Savings percentage: {demand_savings_percent:.2f}%")

    assert abs(
        optimized_demand_cost_eur - 3.79139968
    ) < 0.00000001

    print("Cost check passed.")

In [ ]:
# Reconstruct the decisions behind the minimum cost.
time_slot = 0
remaining = dp_quantities

optimized_schedule_rows = []
completed_quantities = [0] * len(pilot_jobs)

while any(remaining):
    # Candidate decision: wait for five minutes.
    options = [
        (
            minimum_remaining_cost(time_slot + 1, remaining),
            None,
        )
    ]

    # Candidate decisions: produce one unit.
    for job_type, quantity in enumerate(remaining):
        if quantity == 0:
            continue

        next_remaining = list(remaining)
        next_remaining[job_type] -= 1

        total_cost = (
            dp_start_costs[job_type][time_slot]
            + minimum_remaining_cost(
                time_slot + dp_durations[job_type],
                tuple(next_remaining),
            )
        )

        options.append((total_cost, job_type))

    # Choose the decision with the lowest total remaining cost.
    decision_cost, chosen_type = min(
        options, key=lambda option: option[0]
    )

    assert abs(
        decision_cost
        - minimum_remaining_cost(time_slot, remaining)
    ) < 0.00000001

    start_slot = time_slot

    if chosen_type is None:
        time_slot += 1
        activity = "Wait"
        product = ""
        energy_kwh = 0.0
        cost_eur = 0.0

    else:
        template = pilot_jobs[chosen_type]
        profile = dp_profiles[chosen_type]

        completed_quantities[chosen_type] += 1
        activity = (
            f"{template['job_id']}_"
            f"{completed_quantities[chosen_type]:02d}"
        )
        product = template["product"]
        energy_kwh = sum(profile) / 1000
        cost_eur = dp_start_costs[chosen_type][start_slot]

        time_slot += dp_durations[chosen_type]

        next_remaining = list(remaining)
        next_remaining[chosen_type] -= 1
        remaining = tuple(next_remaining)

    start_minutes = 7 * 60 + start_slot * 5
    finish_minutes = 7 * 60 + time_slot * 5

    optimized_schedule_rows.append({
        "activity": activity,
        "product": product,
        "start": (
            f"{start_minutes // 60:02d}:"
            f"{start_minutes % 60:02d}"
        ),
        "finish": (
            f"{finish_minutes // 60:02d}:"
            f"{finish_minutes % 60:02d}"
        ),
        "start_slot": start_slot,
        "finish_slot": time_slot,
        "duration_min": (time_slot - start_slot) * 5,
        "energy_kwh": energy_kwh,
        "cost_eur": cost_eur,
    })

optimized_schedule_df = pd.DataFrame(
    optimized_schedule_rows
)

display(
    optimized_schedule_df.drop(
        columns=["start_slot", "finish_slot"]
    )
)

In [ ]:
schedule_cost = optimized_schedule_df["cost_eur"].sum()
schedule_energy = optimized_schedule_df["energy_kwh"].sum()

waiting_minutes = optimized_schedule_df.loc[
    optimized_schedule_df["activity"] == "Wait",
    "duration_min",
].sum()

# Required product quantities are completed exactly.
assert tuple(completed_quantities) == dp_quantities

# Every activity starts when the previous activity ends.
previous_finish = 0

for row in optimized_schedule_rows:
    assert row["start_slot"] == previous_finish
    assert row["finish_slot"] > row["start_slot"]
    assert row["finish_slot"] <= dp_horizon
    previous_finish = row["finish_slot"]

# Reconstructed totals match the model and the reference energy.
assert abs(
    schedule_cost - optimized_demand_cost_eur
) < 0.00000001

assert abs(
    schedule_energy
    - demand_baseline_df["energy_kwh"].sum()
) < 0.00000001

print("Completed quantities:", completed_quantities)
print(f"Total energy: {schedule_energy:.3f} kWh")
print(f"Total cost: {schedule_cost:.8f} EUR")
print(f"Waiting time: {waiting_minutes} min")
print("Finish:", optimized_schedule_df.iloc[-1]["finish"])
print("All schedule checks passed.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Use the same product colours in both schedules.
product_colours = {
    pilot_jobs[0]["product"]: "#2563EB",
    pilot_jobs[1]["product"]: "#D97706",
    pilot_jobs[2]["product"]: "#059669",
}

wait_colour = "#CBD5E1"


def clock_to_minutes(clock):
    hour, minute = map(int, clock.split(":"))
    return hour * 60 + minute


def draw_schedule(ax, schedule_df, title):
    for _, row in schedule_df.iterrows():
        start = clock_to_minutes(row["start"])
        finish = clock_to_minutes(row["finish"])

        is_wait = row["product"] == ""

        colour = (
            wait_colour
            if is_wait
            else product_colours[row["product"]]
        )

        ax.broken_barh(
            [(start, finish - start)],
            (0, 1),
            facecolors=colour,
            edgecolors="white",
            linewidth=0.6,
        )

    ticks = list(range(7 * 60, 23 * 60 + 1, 60))

    ax.set_xlim(7 * 60, 23 * 60)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [f"{minute // 60:02d}:00" for minute in ticks],
        rotation=45,
    )
    ax.set_yticks([])
    ax.set_title(title, loc="left")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)


fig, axes = plt.subplots(2, 1, figsize=(13, 5))

draw_schedule(
    axes[0],
    demand_baseline_df,
    f"Reference plan — {demand_baseline_cost_eur:.6f} EUR",
)

draw_schedule(
    axes[1],
    optimized_schedule_df,
    f"Optimized plan — {optimized_demand_cost_eur:.6f} EUR",
)

axes[1].set_xlabel("Time of day")

legend_items = [
    Patch(
        facecolor=product_colours[pilot_jobs[0]["product"]],
        label="PES ribbon",
    ),
    Patch(
        facecolor=product_colours[pilot_jobs[1]["product"]],
        label="Braided elastic",
    ),
    Patch(
        facecolor=product_colours[pilot_jobs[2]["product"]],
        label="Smooth strap",
    ),
    Patch(facecolor=wait_colour, label="Waiting"),
]

fig.suptitle(
    "Production schedule comparison — MAQ120, 29 production units"
)

fig.legend(
    handles=legend_items,
    loc="lower center",
    ncol=4,
    frameon=False,
)

fig.tight_layout(rect=[0, 0.10, 1, 0.93])
fig.savefig("demand_schedule_comparison.png", dpi=300)
plt.show()

In [ ]:
from google.colab import files

files.download("demand_schedule_comparison.png")

In [ ]:
def count_product_changes(schedule_df):
    # Waiting does not change which product was last produced.
    products = schedule_df.loc[
        schedule_df["product"] != "",
        "product",
    ].tolist()

    return sum(
        previous != current
        for previous, current in zip(products, products[1:])
    )


reference_changes = count_product_changes(
    demand_baseline_df
)

optimized_changes = count_product_changes(
    optimized_schedule_df
)

print("Reference plan product changes:", reference_changes)
print("Optimized plan product changes:", optimized_changes)

In [ ]:
setup_rows = []

production_minutes = demand_baseline_df.shape[0]
production_minutes = sum(
    len(job["energy_profile_wh"]) * 5
    for job in production_jobs
)

for setup_minutes in [0, 5, 10, 15]:
    reference_required = (
        production_minutes
        + reference_changes * setup_minutes
    )

    optimized_required = (
        production_minutes
        + optimized_changes * setup_minutes
    )

    setup_rows.append({
        "setup_min_per_change": setup_minutes,
        "reference_required_min": reference_required,
        "optimized_required_min": optimized_required,
        "reference_capacity_margin_min": 960 - reference_required,
        "optimized_capacity_margin_min": 960 - optimized_required,
    })

setup_capacity_df = pd.DataFrame(setup_rows)
display(setup_capacity_df)

In [ ]:
from functools import lru_cache
from math import inf, isfinite

setup_minutes = 5
setup_slots = setup_minutes // 5


@lru_cache(maxsize=None)
def minimum_cost_with_setup(time_slot, remaining, last_type):
    if not any(remaining):
        return 0.0

    required_processing_slots = sum(
        quantity * duration
        for quantity, duration in zip(
            remaining, dp_durations
        )
    )

    if time_slot + required_processing_slots > dp_horizon:
        return inf

    # Waiting preserves the last product type.
    best_cost = minimum_cost_with_setup(
        time_slot + 1, remaining, last_type
    )

    for job_type, quantity in enumerate(remaining):
        if quantity == 0:
            continue

        change_required = (
            last_type != -1 and last_type != job_type
        )

        preparation_slots = (
            setup_slots if change_required else 0
        )

        production_start = time_slot + preparation_slots
        production_finish = (
            production_start + dp_durations[job_type]
        )

        if production_finish > dp_horizon:
            continue

        next_remaining = list(remaining)
        next_remaining[job_type] -= 1

        candidate_cost = (
            dp_start_costs[job_type][production_start]
            + minimum_cost_with_setup(
                production_finish,
                tuple(next_remaining),
                job_type,
            )
        )

        best_cost = min(best_cost, candidate_cost)

    return best_cost


setup_optimized_cost = minimum_cost_with_setup(
    0, dp_quantities, -1
)

# Recalculate the original reference order with the same setup rule.
reference_slot = 0
reference_last_mode = None
setup_reference_cost = 0.0

for job in production_jobs:
    if (
        reference_last_mode is not None
        and reference_last_mode != job["mode"]
    ):
        reference_slot += setup_slots

    profile = job["energy_profile_wh"]

    assert reference_slot + len(profile) <= dp_horizon

    setup_reference_cost += sum(
        energy_wh * dp_prices[reference_slot + offset]
        for offset, energy_wh in enumerate(profile)
    )

    reference_slot += len(profile)
    reference_last_mode = job["mode"]

if isfinite(setup_optimized_cost):
    setup_saving = setup_reference_cost - setup_optimized_cost
    setup_saving_percent = (
        setup_saving / setup_reference_cost * 100
    )

    print(f"Setup per product change: {setup_minutes} min")
    print(f"Reference cost: {setup_reference_cost:.8f} EUR")
    print(f"Optimized cost: {setup_optimized_cost:.8f} EUR")
    print(f"Savings: {setup_saving:.8f} EUR")
    print(f"Savings percentage: {setup_saving_percent:.2f}%")

    assert abs(
        setup_optimized_cost - 3.79714186
    ) < 0.00000001

    print("Cost check passed.")
else:
    print("No feasible schedule.")

In [ ]:
setup_schedule_rows = []
setup_completed = [0] * len(pilot_jobs)


def add_schedule_row(activity, product, start, finish,
                     energy_kwh=0.0, cost_eur=0.0):
    start_minutes = 7 * 60 + start * 5
    finish_minutes = 7 * 60 + finish * 5

    setup_schedule_rows.append({
        "activity": activity,
        "product": product,
        "start": (
            f"{start_minutes // 60:02d}:"
            f"{start_minutes % 60:02d}"
        ),
        "finish": (
            f"{finish_minutes // 60:02d}:"
            f"{finish_minutes % 60:02d}"
        ),
        "start_slot": start,
        "finish_slot": finish,
        "duration_min": (finish - start) * 5,
        "energy_kwh": energy_kwh,
        "cost_eur": cost_eur,
    })


time_slot = 0
remaining = dp_quantities
last_type = -1

while any(remaining):
    # Option: wait for five minutes.
    options = [(
        minimum_cost_with_setup(
            time_slot + 1, remaining, last_type
        ),
        None,
        time_slot,
        time_slot + 1,
        remaining,
    )]

    # Options: prepare if necessary, then produce one unit.
    for job_type, quantity in enumerate(remaining):
        if quantity == 0:
            continue

        change_required = (
            last_type != -1 and last_type != job_type
        )

        production_start = time_slot + (
            setup_slots if change_required else 0
        )

        production_finish = (
            production_start + dp_durations[job_type]
        )

        if production_finish > dp_horizon:
            continue

        next_remaining = list(remaining)
        next_remaining[job_type] -= 1
        next_remaining = tuple(next_remaining)

        cost = (
            dp_start_costs[job_type][production_start]
            + minimum_cost_with_setup(
                production_finish,
                next_remaining,
                job_type,
            )
        )

        options.append((
            cost,
            job_type,
            production_start,
            production_finish,
            next_remaining,
        ))

    score, chosen_type, start, finish, next_remaining = min(
        options, key=lambda option: option[0]
    )

    assert abs(
        score - minimum_cost_with_setup(
            time_slot, remaining, last_type
        )
    ) < 0.00000001

    if chosen_type is None:
        add_schedule_row("Wait", "", time_slot, finish)

    else:
        product = pilot_jobs[chosen_type]["product"]

        if start > time_slot:
            add_schedule_row(
                "Setup", product, time_slot, start
            )

        add_schedule_row(
            "Production",
            product,
            start,
            finish,
            sum(dp_profiles[chosen_type]) / 1000,
            dp_start_costs[chosen_type][start],
        )

        setup_completed[chosen_type] += 1
        last_type = chosen_type

    time_slot = finish
    remaining = next_remaining

setup_schedule_df = pd.DataFrame(setup_schedule_rows)

display(
    setup_schedule_df.drop(
        columns=["start_slot", "finish_slot"]
    )
)

In [ ]:
# Check quantities and timeline.
assert tuple(setup_completed) == dp_quantities

previous_finish = 0

for row in setup_schedule_rows:
    assert row["start_slot"] == previous_finish
    assert row["finish_slot"] > row["start_slot"]
    assert row["finish_slot"] <= dp_horizon
    previous_finish = row["finish_slot"]

# Check the reconstructed cost and energy.
assert abs(
    setup_schedule_df["cost_eur"].sum()
    - setup_optimized_cost
) < 0.00000001

assert abs(
    setup_schedule_df["energy_kwh"].sum()
    - demand_baseline_df["energy_kwh"].sum()
) < 0.00000001

print("Completed quantities:", setup_completed)

print("\nMinutes by activity:")
print(
    setup_schedule_df.groupby("activity")["duration_min"].sum()
)

print(
    f"\nTotal cost: "
    f"{setup_schedule_df['cost_eur'].sum():.8f} EUR"
)

print("Finish:", setup_schedule_df.iloc[-1]["finish"])
print("All checks passed.")

In [ ]:
from math import inf, isfinite


def reference_cost_for_setup(preparation_slots):
    current_slot = 0
    previous_mode = None
    total_cost = 0.0

    for job in production_jobs:
        if (
            previous_mode is not None
            and previous_mode != job["mode"]
        ):
            current_slot += preparation_slots

        profile = job["energy_profile_wh"]

        if current_slot + len(profile) > dp_horizon:
            return inf

        total_cost += sum(
            energy_wh * dp_prices[current_slot + offset]
            for offset, energy_wh in enumerate(profile)
        )

        current_slot += len(profile)
        previous_mode = job["mode"]

    return total_cost


setup_sensitivity_rows = []
original_setup_slots = setup_slots

try:
    for scenario_minutes in [0, 5, 10, 15]:
        setup_slots = scenario_minutes // 5

        # Different setup assumptions require fresh calculations.
        minimum_cost_with_setup.cache_clear()

        scenario_optimized_cost = minimum_cost_with_setup(
            0, dp_quantities, -1
        )

        scenario_reference_cost = reference_cost_for_setup(
            setup_slots
        )

        comparable = (
            isfinite(scenario_optimized_cost)
            and isfinite(scenario_reference_cost)
        )

        setup_sensitivity_rows.append({
            "setup_minutes": scenario_minutes,
            "reference_cost_eur": (
                scenario_reference_cost
                if isfinite(scenario_reference_cost)
                else None
            ),
            "optimized_cost_eur": (
                scenario_optimized_cost
                if isfinite(scenario_optimized_cost)
                else None
            ),
            "savings_eur": (
                scenario_reference_cost - scenario_optimized_cost
                if comparable else None
            ),
            "savings_percent": (
                (scenario_reference_cost - scenario_optimized_cost)
                / scenario_reference_cost * 100
                if comparable else None
            ),
            "optimized_feasible": isfinite(
                scenario_optimized_cost
            ),
        })

finally:
    # Restore the original setup assumption.
    setup_slots = original_setup_slots
    minimum_cost_with_setup.cache_clear()

    minimum_cost_with_setup(
        0, dp_quantities, -1
    )


setup_sensitivity_df = pd.DataFrame(
    setup_sensitivity_rows
)

display(
    setup_sensitivity_df.style.format({
        "reference_cost_eur": "{:.6f}",
        "optimized_cost_eur": "{:.6f}",
        "savings_eur": "{:.6f}",
        "savings_percent": "{:.2f}",
    })
)

In [ ]:
def summarize_setup_decisions():
    time_slot = 0
    remaining = dp_quantities
    last_type = -1

    changes = 0
    waiting_slots = 0
    reconstructed_cost = 0.0
    completed = [0] * len(pilot_jobs)

    while any(remaining):
        options = [(
            minimum_cost_with_setup(
                time_slot + 1, remaining, last_type
            ),
            None,
            time_slot,
            time_slot + 1,
            remaining,
        )]

        for job_type, quantity in enumerate(remaining):
            if quantity == 0:
                continue

            needs_setup = (
                last_type != -1 and last_type != job_type
            )

            start = time_slot + (
                setup_slots if needs_setup else 0
            )
            finish = start + dp_durations[job_type]

            if finish > dp_horizon:
                continue

            next_remaining = list(remaining)
            next_remaining[job_type] -= 1
            next_remaining = tuple(next_remaining)

            cost = (
                dp_start_costs[job_type][start]
                + minimum_cost_with_setup(
                    finish, next_remaining, job_type
                )
            )

            options.append((
                cost, job_type, start, finish, next_remaining
            ))

        score, chosen_type, start, finish, next_remaining = min(
            options, key=lambda option: option[0]
        )

        assert abs(
            score - minimum_cost_with_setup(
                time_slot, remaining, last_type
            )
        ) < 0.00000001

        if chosen_type is None:
            waiting_slots += 1
        else:
            if last_type != -1 and last_type != chosen_type:
                changes += 1

            completed[chosen_type] += 1
            reconstructed_cost += dp_start_costs[chosen_type][start]
            last_type = chosen_type

        time_slot = finish
        remaining = next_remaining

    assert tuple(completed) == dp_quantities
    assert abs(
        reconstructed_cost
        - minimum_cost_with_setup(0, dp_quantities, -1)
    ) < 0.00000001

    return {
        "product_changes": changes,
        "total_setup_min": changes * setup_slots * 5,
        "waiting_min": waiting_slots * 5,
        "finish_time": (
            f"{(420 + time_slot * 5) // 60:02d}:"
            f"{(420 + time_slot * 5) % 60:02d}"
        ),
    }


decision_rows = []
saved_setup_slots = setup_slots

try:
    for scenario_minutes in [0, 5, 10, 15]:
        setup_slots = scenario_minutes // 5
        minimum_cost_with_setup.cache_clear()

        minimum_cost_with_setup(0, dp_quantities, -1)
        summary = summarize_setup_decisions()

        summary["setup_minutes"] = scenario_minutes
        decision_rows.append(summary)

finally:
    setup_slots = saved_setup_slots
    minimum_cost_with_setup.cache_clear()
    minimum_cost_with_setup(0, dp_quantities, -1)


setup_decisions_df = pd.DataFrame(decision_rows)

setup_analysis_df = setup_sensitivity_df.merge(
    setup_decisions_df,
    on="setup_minutes",
    validate="one_to_one",
)

display(setup_analysis_df)

In [ ]:
from pathlib import Path
import json
import zipfile
from google.colab import files

output_folder = Path("checkpoint_results")
output_folder.mkdir(exist_ok=True)

tables = {
    "reference_schedule.csv": demand_baseline_df,
    "optimized_schedule_no_setup.csv": optimized_schedule_df,
    "optimized_schedule_5min_setup.csv": setup_schedule_df,
    "setup_sensitivity_analysis.csv": setup_analysis_df,
}

for filename, table in tables.items():
    table.to_csv(output_folder / filename, index=False)

# Record the assumptions behind these results.
assumptions = {
    "machine": "MAQ120",
    "products": [job["product"] for job in pilot_jobs],
    "quantities": list(dp_quantities),
    "working_hours": "07:00–23:00",
    "time_step_minutes": 5,
    "price_period": "First working day of the source dataset",
    "price_information": "Full-day prices known before scheduling",
    "setup_scenarios_minutes": [0, 5, 10, 15],
    "initial_setup_minutes": 0,
    "setup_energy_kwh": 0,
    "waiting_energy_kwh": 0,
    "all_jobs_available_at": "07:00",
    "jobs_interruptible": False,
    "demand_timing": (
        "Selected source quantities assigned to one day "
        "as a modelling scenario"
    ),
}

with open(
    output_folder / "assumptions.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(assumptions, file, indent=2, ensure_ascii=False)

archive_name = "energy_scheduling_checkpoint.zip"

with zipfile.ZipFile(archive_name, "w") as archive:
    for filename in tables:
        path = output_folder / filename
        archive.write(path, arcname=path.name)

    path = output_folder / "assumptions.json"
    archive.write(path, arcname=path.name)

files.download(archive_name)